In [1]:
# ============================================================================
# INPUT CHECK - runs first, before the stage below it.
#
# Every stage resolves its inputs with glob("/kaggle/input/**/<name>") and takes
# the FIRST hit alphabetically. A file attached twice is therefore not an error,
# it is a coin toss - and the order is worse than random: a folder named
# stage1-data-pipeline-OLD sorts BEFORE stage1-data-pipeline, so the copy you
# meant to retire is the one that wins, silently.
#
# This prints what will actually be read, and stops the run if anything is
# attached twice. Everything lives inside one function so it cannot collide
# with a name the stage below uses.
# ============================================================================
def _check_kaggle_inputs():
    import glob, os
    from datetime import datetime

    WANTED = {
        "unified.parquet":                     "Stage 1 corpus        (Stages 2, 4)",
        "splits.json":                         "Stage 1 splits        (Stage 2)",
        "predictions_finetuned.parquet":       "Stage 2 encoder store (Stage 2 resume, 4, 5)",
        "predictions_llm.parquet":             "Stage 3 LLM binary    (Stages 4, 5)",
        "predictions_subtype.parquet":         "Stage 3b raw subtype  (Stage 3b-repair)",
        "predictions_subtype_repaired.parquet": "Stage 3b repaired    (Stages 4, 5)",
        "predictions_llm_repaired.parquet":     "Stage 3b R8 binary   (Stages 4, 5)",
        "tab9_evaluability.csv":               "Stage 4 gate          (Stage 5)",
    }
    # Files THIS stage actually reads. A bad version of one of these is a
    # hard stop; anything else in WANTED is checked for information only.
    CONSUMED = {"unified.parquet", "predictions_finetuned.parquet",
                "predictions_llm_repaired.parquet",
                "predictions_subtype_repaired.parquet"}

    print("=" * 78)
    print("ATTACHED INPUTS")
    print("=" * 78)
    problems = []
    for fname, used_by in WANTED.items():
        hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
        print(f"\n{fname}   <- {used_by}")
        if not hits:
            print("   (not attached)")
            continue
        for i, h in enumerate(hits):
            when = datetime.fromtimestamp(os.path.getmtime(h)).strftime("%Y-%m-%d %H:%M")
            mark = "  >> THIS ONE WILL BE USED" if i == 0 else "     ignored"
            print(f"   {h}\n      {os.path.getsize(h):>12,} bytes   {when}{mark}")
        if len(hits) > 1:
            problems.append(f"DUPLICATE: {fname} is attached {len(hits)} times")

    print("\n" + "=" * 78)
    print("ROW COUNTS OF WHAT WILL ACTUALLY BE READ")
    print("=" * 78)
    # A file of the right NAME can still be the wrong VERSION. The row count is
    # what tells them apart. The encoder store has two legitimate sizes
    # depending on where you are in the chain, so it is checked against both.
    EXPECTED = {
        "unified.parquet":                     ({1412}, "1,412"),
        "predictions_finetuned.parquet":       ({16900, 24280},
                                                "16,900 before Stage 2 / 24,280 after it"),
        "predictions_llm.parquet":             ({35049}, "35,049"),
        "predictions_subtype_repaired.parquet": ({16786}, "16,786"),
        "predictions_llm_repaired.parquet":     ({35049}, "35,049"),
    }
    try:
        import pandas as pd
        for fname, (ok_values, note) in EXPECTED.items():
            hits = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
            if not hits:
                continue
            n = len(pd.read_parquet(hits[0]))
            # The expected values are the counts of the COMMITTED REFERENCE
            # run. A LARGER prediction store is legitimate after an API top-up
            # (the harnesses resume and add previously rate-limited items); a
            # SMALLER one is an old or partial store and must not be used.
            # unified.parquet must match exactly: requirement ids are
            # positional, so a corpus of any other size silently re-points
            # every stored prediction.
            if n in ok_values:
                flag = "OK"
            elif fname == "unified.parquet":
                flag = "<-- CORPUS SIZE CHANGED: ids are positional, do NOT run"
            elif n > max(ok_values):
                flag = ("larger than the committed reference (expected after an "
                        "API top-up; verify provenance, then update EXPECTED here)")
            else:
                flag = ("<-- SMALLER than the committed reference "
                        "(old/partial store), do not run")
            print(f"  {fname:38s} {n:>7,} rows   expected {note}   {flag}")
            # A '<--' flag on a file THIS stage actually consumes is a hard
            # stop: the old check printed 'do not run' and then ran anyway,
            # which is how hours of compute get spent against a wrong input.
            if flag.startswith("<--") and fname in CONSUMED:
                problems.append(f"BAD INPUT: {fname} - {flag.lstrip('<- ')}")
            if fname == "predictions_finetuned.parquet":
                which = ("the OLD store (correct INPUT for Stage 2 itself)" if n == 16900
                         else "the NEW store (required by Stages 4 and 5)" if n == 24280
                         else "neither size this chain produces")
                print(f"      -> this is {which}")
    except Exception as e:
        print("  (could not read:", e, ")")

    sp = sorted(glob.glob("/kaggle/input/**/splits.json", recursive=True))
    if sp:
        import json
        fams = json.load(open(sp[0]))
        n_folds = sum(len(v) for v in fams.values())
        verdict = "OK" if len(fams) == 14 else "<-- OLD Stage 1 output, re-run Stage 1 first"
        print(f"\n  splits.json: {len(fams)} families, {n_folds} folds   {verdict}")
        if verdict.startswith("<--") and "splits.json" in CONSUMED:
            problems.append("BAD INPUT: splits.json - OLD Stage 1 output "
                            f"({len(fams)} families, expected 14)")

    # Stage 4 and Stage 5 PREFER predictions_llm_repaired over the harness
    # store, and fall back silently when it is absent. Absent means an older
    # stage3b-repair version is attached - one from before R8 existed - and the
    # analysis would then run on the longest-first parse the project moved off.
    # The fallback is by design; being unaware of it is not.
    _raw = sorted(glob.glob("/kaggle/input/**/predictions_llm.parquet", recursive=True))
    _rep = sorted(glob.glob("/kaggle/input/**/predictions_llm_repaired.parquet",
                            recursive=True))
    if _raw and not _rep:
        print("  WARNING: predictions_llm.parquet is attached but")
        print("           predictions_llm_repaired.parquet is NOT.")
        print("           Stages 4 and 5 will silently fall back to the")
        print("           unrepaired binary store. Attach the stage3b-repair")
        print("           version that printed an [R8] line. Harmless for")
        print("           Stage 3b-repair itself, which produces that file.")

    print("\n" + "=" * 78)
    if problems:
        for p in problems:
            print(f"  {p}")
        raise SystemExit("Fix the problems above (detach duplicates / attach "
                         "the right versions), then run again. "
                         "The stage below did NOT run.")
    print("No duplicates. Running the stage now.")
    print("=" * 78 + "\n")


_check_kaggle_inputs()


# =============================================================================
# STAGE 4 - ANALYSIS  (binary + NFR sub-type, RQ1 / RQ2)
#
# INPUTS (add all four as Notebook-Output datasets):
#   stage1-data-pipeline        -> unified corpus (class balance for the manifest)
#   stage2-finetuned-baselines  -> predictions_finetuned.parquet
#   stage3-llm-harness          -> predictions_llm.parquet
#   stage3b-repair              -> predictions_subtype_repaired.parquet
#                                  (or stage3b-topup / stage3b-subtype-harness)
#
# SETTINGS: Accelerator = None, Internet = Off.
#
# OUTPUTS -> /kaggle/working/
#   tab1_main_in_domain.csv / .tex        tab2_main_transfer.csv / .tex
#   tab3_generalisation_gap.csv           tab4_mcnemar.csv
#   tab5_subtype.csv                      tab6_perclass.csv
#   tab7_fewshot.csv                      tab8_prompt_sensitivity.csv
#   tab9_evaluability.csv                 tab0_baselines.csv
#   tab1w / tab2w (weighted-F1)           fig1..fig6 (.png + .pdf)
#   stage4_provenance.json
#
# -----------------------------------------------------------------------------
# REVISION 2  -  what changed and why  (search the file for the [FIX-n] tags)
#
# [FIX-1]  CRITICAL.  The headline filter read
#              head = store[store.prompt_id.isin([...]) & (store.shot_k == 0)]
#          Every fine-tuned row is stored with shot_k = -1 and
#          prompt_id = "supervised" (deliberately: the sentinel "n/a" would be
#          silently coerced to NaN by pandas).  BOTH conditions therefore
#          excluded all 16,900 encoder rows, and the consequences cascaded:
#            - tab1 and tab2 came out byte-identical (md5 d4ad0dfe...), because
#              the only regime surviving in both panels was "prompted";
#            - tab3 contained no encoder row, so RQ2 had one side only;
#            - tab4_mcnemar.csv was empty and provenance recorded
#              "mcnemar_tests": 0, so no comparison carried a p-value.
#          head now keeps every encoder row unconditionally, and a hard guard
#          below aborts the run if the encoder count is ever zero again.
#
# [FIX-2]  RQ1 asks for "macro- and weighted-F1".  weighted_f1 was computed in
#          cell() but never surfaced; tab1w / tab2w now emit it.
#
# [FIX-3]  tab3 now names the regime the drop was measured against
#          (cross_project vs cross_dataset) instead of silently taking the max
#          over both - the thesis needs to distinguish the two.
#
# [FIX-4]  tab4 is now written with its column header even when empty, and the
#          run prints the test count, so a silent zero can never pass unnoticed.
#
# NOTE on tab0: with encoders present the gold set per (task, dataset) is the
# full corpus (968 / 968 / 444) rather than the LLM frame (960 / 960 / 436),
# because the 8 items carved out as few-shot exemplars are still scored by the
# encoders.  The majority-class baseline therefore shifts by a few thousandths.
# This is the correct definition of the baseline; quote the new numbers.
# =============================================================================

import glob
import json
import os
import re
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy import stats as sps
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support)

warnings.filterwarnings("ignore")
pd.set_option("display.width", 250)

# ----------------------------------------------------------------- configuration
OUT = "/kaggle/working"
SEED = 42
N_BOOT = 2000
MIN_CLASS_SUPPORT = 5      # a class below this cannot carry 1/K of a macro average
MIN_PARSE_RATE = 0.50      # below this the cell was not measured, only failed
MIN_N = 40                 # below this no macro-F1 is reported at all
ALPHA = 0.05
UNPARSED = "__unparsed__"

os.makedirs(OUT, exist_ok=True)

CATEGORIES_ALL = ["availability", "fault_tolerance", "legal", "look_and_feel",
                  "maintainability", "operational", "performance", "portability",
                  "scalability", "security", "usability"]
TOP4 = ["security", "usability", "operational", "performance"]
TOP6 = TOP4 + ["look_and_feel", "availability"]

LABELSETS = {
    "fr_nfr":       ["FR", "NFR"],
    "security":     ["security", "non-security"],
    "subtype_all":  CATEGORIES_ALL,
    "subtype_top6": TOP6,
    "subtype_top4": TOP4,
}
BINARY_TASKS = ["fr_nfr", "security"]
SUBTYPE_TASKS = ["subtype_top4", "subtype_top6", "subtype_all"]

TASK_LABEL = {"fr_nfr": "FR/NFR", "security": "Security",
              "subtype_top4": "NFR sub-types (top-4)",
              "subtype_top6": "NFR sub-types (top-6)",
              "subtype_all": "NFR sub-types (all-11)"}
TIER_LABEL = {"finetuned": "Fine-tuned encoder",
              "open_local": "LLM - local open-weight",
              "open_hosted": "LLM - hosted open-weight (API)",
              "commercial": "LLM - commercial closed-weight (API)"}
TIER_COLOR = {"finetuned": "#0F766E", "open_local": "#2563EB",
              "open_hosted": "#7C3AED", "commercial": "#C026D3",
              "baseline": "#9CA3AF"}

PRETTY = {
    "bert-base-uncased-weighted": "BERT (weighted)",
    "bert-base-uncased-unweighted": "BERT (unweighted)",
    "roberta-base-weighted": "RoBERTa (weighted)",
    "roberta-base-unweighted": "RoBERTa (unweighted)",
    "qwen2.5-7b-instruct": "Qwen2.5-7B",
    "qwen2.5-3b-instruct": "Qwen2.5-3B",
    "llama-3.1-8b-instruct": "Llama-3.1-8B",
    "phi-4-mini-instruct": "Phi-4-mini",
    "gemma-2-2b-it": "Gemma-2-2B",
    "smollm3-3b": "SmolLM3-3B",
    "groq-llama-3.3-70b-versatile": "Llama-3.3-70B (Groq)",
    "gemini-3.1-flash-lite": "Gemini-3.1-Flash-Lite",
}


def pretty(tag):
    if tag in PRETTY:
        return PRETTY[tag]
    t = re.sub(r"^openrouter-", "", str(tag))
    return t.replace("-instruct", "").replace("-", " ").title()


# ------------------------------------------------------------------------- io
def find(*patterns):
    for p in patterns:
        hits = sorted(glob.glob(p, recursive=True))
        if hits:
            return hits[0]
    return None


def read_any(path):
    if path is None:
        return None
    df = (pd.read_parquet(path) if path.endswith(".parquet")
          else pd.read_csv(path, keep_default_na=False, low_memory=False))
    return df


SOURCES = {
    "finetuned": find("/kaggle/input/**/predictions_finetuned.parquet",
                      "/kaggle/input/**/predictions_finetuned.csv",
                      f"{OUT}/predictions_finetuned.parquet"),
    # Repaired store FIRST, for the same reason the sub-type one is: it carries
    # y_pred_strict and the positional parse this project standardised on. The
    # harness store still holds the older longest-first parse, which differs on
    # 17 predictions. Raw store last, as the fallback it is.
    "llm_binary": find("/kaggle/input/**/predictions_llm_repaired.parquet",
                       "/kaggle/input/**/predictions_llm_repaired.csv",
                       "/kaggle/input/**/predictions_llm.parquet",
                       "/kaggle/input/**/predictions_llm.csv",
                       f"{OUT}/predictions_llm_repaired.parquet",
                       f"{OUT}/predictions_llm.parquet"),
    # Repaired store FIRST - it is the one that carries in_core and
    # y_pred_strict, both of which this notebook now uses. The comment used to
    # say "repaired store first" while the code searched for a `_topup` file
    # ahead of it; no notebook in this project produces `_topup`, so the first
    # pattern could only ever miss. Raw store last, as the fallback it is.
    "llm_subtype": find("/kaggle/input/**/predictions_subtype_repaired.parquet",
                        "/kaggle/input/**/predictions_subtype_repaired.csv",
                        "/kaggle/input/**/predictions_subtype.parquet",
                        "/kaggle/input/**/predictions_subtype.csv"),
}

frames, provenance = [], {}
for name, path in SOURCES.items():
    if path is None:
        print(f"[warn] {name}: NOT FOUND - continuing without it")
        continue
    df = read_any(path)
    df["_source"] = name
    frames.append(df)
    provenance[name] = {"path": path, "rows": int(len(df))}
    print(f"[load] {name:12s} {len(df):>7,} rows  {path}")

if not frames:
    raise FileNotFoundError("No prediction stores found. Check notebook Inputs.")

store = pd.concat(frames, ignore_index=True, sort=False)

# --- normalise -----------------------------------------------------------------
for c in ["parse_ok"]:
    if c not in store.columns:
        store[c] = True
# Strictly ["true", "1"], matching Stage 5. Accepting "nan" and "" as True meant
# any row that lost the column counted as SUCCESSFULLY PARSED - the opposite of
# the strict convention this notebook declares in its provenance block, and the
# opposite of what Stage 5 does with the same data.
store["parse_ok"] = store["parse_ok"].astype(str).str.lower().isin(["true", "1"])
store.loc[store.approach.astype(str) == "finetuned", "parse_ok"] = True
store["shot_k"] = pd.to_numeric(store.get("shot_k", 0), errors="coerce").fillna(0).astype(int)
for c in ["prompt_id", "split", "eval_regime", "model_type", "task", "dataset",
          "approach", "y_true", "y_pred", "id", "model_tag"]:
    store[c] = store[c].astype(str)
store["model"] = store.model_tag.map(pretty)
store["tier"] = np.where(store.approach == "finetuned", "finetuned", store.model_type)

# strict prediction: an unparseable answer is wrong, not absent
store["y_hat"] = store.y_pred.where(store.parse_ok, UNPARSED)

# The repaired sub-type store already carries y_pred_strict under exactly this
# convention. Honour it where it exists instead of silently recomputing: if the
# repair cell adopts the positional parse (APPLY_REPARSE), that decision has to
# reach the analysis, and recomputing here would quietly discard it.
if "y_pred_strict" in store.columns:
    # Both REPAIRED LLM stores carry this column; the encoder store does not,
    # so after the concat it is missing on those rows. Test for that with
    # notna() BEFORE any string coercion: on pandas 3 the new string dtype
    # keeps NA through .astype(str) and an "is it the text 'nan'" comparison
    # returns NA, not False - which silently adopted the empty value on every
    # row and turned y_hat into a missing label across the whole store.
    _present = store["y_pred_strict"].notna()
    _sv = store["y_pred_strict"].where(_present, "").astype(str)
    _has = (_present & _sv.str.strip().ne("")
            & _sv.str.lower().ne("nan")).fillna(False).astype(bool)
    _diff = int((_has & (_sv != store["y_hat"])).sum())
    store.loc[_has, "y_hat"] = _sv[_has]
    print(f"[strict] adopted y_pred_strict on {int(_has.sum()):,} row(s); "
          f"{_diff} differed from the value recomputed here")

# `in_core` marks the shared CORE subset, and only the repaired SUB-TYPE store
# carries it. Binary rows are set True - not because every model saw the same
# frame (they did not: ~960 items for a local model, ~290 for Groq, 60 for
# Gemini, 21 for the OpenRouter route), but because no core/full distinction
# was ever recorded for the binary tasks, so there is no subset to restrict to.
# The like-for-like tables do not rely on this flag: they intersect the ids
# each model actually answered, which handles the unequal frames directly.
if "in_core" not in store.columns:
    store["in_core"] = True
# The fill must happen BEFORE the string coercion, and the guard above is not
# enough on its own. Only the repaired sub-type store carries in_core, so after
# the concat the COLUMN exists while 51,949 rows hold NaN - the guard never
# fires, and astype(str) turns those NaN into the literal "nan", which is not in
# ["true","1"], so every encoder row silently became out-of-core. tab10 then did
# `g[g.in_core] if g.in_core.any() else g`: on the binary tasks nothing was
# in_core so the fallback saved the table, but on the three sub-type tasks the
# LLM rows made .any() True and every fine-tuned baseline vanished from the
# like-for-like ranking - the one table whose entire purpose is a fair
# comparison. RoBERTa scores 0.9660 on the top-4 intersection against the best
# LLM's 0.9298, so the omission inverted the winner in all three blocks.
store["in_core"] = (store["in_core"].where(store["in_core"].notna(), True)
                    .astype(str).str.lower().isin(["true", "1"]))

print(f"[store] {len(store):,} rows | tasks {sorted(store.task.unique())} | "
      f"regimes {sorted(store.eval_regime.unique())}")


# ============================================================================
# metrics
# ============================================================================
def _macro_f1_fast(t_idx, p_idx, K):
    if t_idx.ndim == 1:
        t_idx, p_idx = t_idx[None, :], p_idx[None, :]
    B = t_idx.shape[0]
    off = np.arange(B)[:, None] * (K + 1)
    tp = np.bincount((t_idx + off)[t_idx == p_idx].ravel(),
                     minlength=B * (K + 1)).reshape(B, K + 1)[:, :K]
    pred = np.bincount((p_idx + off).ravel(), minlength=B * (K + 1)).reshape(B, K + 1)[:, :K]
    act = np.bincount((t_idx + off).ravel(), minlength=B * (K + 1)).reshape(B, K + 1)[:, :K]
    den = pred + act
    return np.divide(2.0 * tp, den, out=np.zeros_like(den, dtype=float),
                     where=den > 0).mean(axis=1)


def strat_boot_ci(y_true, y_hat, labelset, n_boot=N_BOOT, seed=SEED):
    """Stratified bootstrap: resample within each true class so the label
    distribution - the thing macro-F1 averages over - is held fixed. An i.i.d.
    bootstrap can lose a rare class entirely, score it 0, and still divide by K."""
    y_true, y_hat = np.asarray(y_true), np.asarray(y_hat)
    if len(y_true) < 10:
        return np.nan, np.nan
    K = len(labelset)
    code = {c: i for i, c in enumerate(labelset)}
    t = np.array([code.get(v, K) for v in y_true])
    p = np.array([code.get(v, K) for v in y_hat])
    rng = np.random.default_rng(seed)
    groups = [np.flatnonzero(t == i) for i in range(K)]
    groups = [g for g in groups if len(g)]
    take = np.concatenate([rng.choice(g, size=(n_boot, len(g)), replace=True)
                           for g in groups], axis=1)
    s = _macro_f1_fast(t[take], p[take], K)
    lo, hi = np.percentile(s, [2.5, 97.5])
    return round(float(lo), 4), round(float(hi), 4)


def evaluability(y_true, labelset):
    cnt = pd.Series(list(y_true)).value_counts()
    sup = {c: int(cnt.get(c, 0)) for c in labelset}
    bad = {c: n for c, n in sup.items() if n < MIN_CLASS_SUPPORT}
    return (len(bad) == 0), min(sup.values()), bad


def cell(g, labelset):
    yt, yp = g.y_true.to_numpy(), g.y_hat.to_numpy()
    lo, hi = strat_boot_ci(yt, yp, labelset)
    ok, min_sup, bad = evaluability(yt, labelset)
    parse_rate = float(g.parse_ok.mean())
    reportable = bool(ok and len(g) >= MIN_N and parse_rate >= MIN_PARSE_RATE)
    reasons = []
    if not ok:
        reasons.append(f"class support<{MIN_CLASS_SUPPORT}")
    if len(g) < MIN_N:
        reasons.append(f"n<{MIN_N}")
    if parse_rate < MIN_PARSE_RATE:
        reasons.append(f"parse<{MIN_PARSE_RATE}")
    return {
        "n": len(g), "n_parsed": int(g.parse_ok.sum()),
        "parse_rate": round(parse_rate, 4),
        "macro_f1": round(float(f1_score(yt, yp, labels=labelset,
                                         average="macro", zero_division=0)), 4),
        "weighted_f1": round(float(f1_score(yt, yp, labels=labelset,
                                            average="weighted", zero_division=0)), 4),
        "accuracy": round(float(accuracy_score(yt, yp)), 4),
        "ci_lo": lo, "ci_hi": hi,
        "ci_width": round(hi - lo, 4) if not np.isnan(hi) else np.nan,
        "min_class_support": min_sup,
        "reportable": reportable,
        "exclusion_reason": "; ".join(reasons) or "-",
        "underpowered_classes": ";".join(f"{k}={v}" for k, v in sorted(bad.items())) or "-",
    }


# ---------------------------------------------------------------------- [FIX-1]
# Headline view: base prompt and zero-shot for the LLMs; EVERY row for the
# encoders.  A fine-tuned run has no prompt and no shot count, so it must not be
# filtered on either. Encoder rows carry shot_k = -1 and prompt_id = "supervised".
IS_ENCODER = store.approach.astype(str).str.lower() == "finetuned"
IS_HEADLINE_PROMPT = (store.prompt_id.isin(["base", "nan", "", "none"])
                      & (store.shot_k == 0))
head = store[IS_ENCODER | IS_HEADLINE_PROMPT].copy()

# Guard. This is the check that would have caught the original defect: if the
# encoders vanish, RQ2 and RQ3 are unanswerable and every downstream table is
# quietly wrong rather than visibly missing.
_n_enc = int((head.approach.astype(str).str.lower() == "finetuned").sum())
_n_llm = int(len(head) - _n_enc)
print(f"[head] {len(head):,} rows | encoder {_n_enc:,} | prompted LLM {_n_llm:,}")
if _n_enc == 0:
    raise RuntimeError(
        "head contains ZERO fine-tuned encoder rows. RQ2 (generalisation gap) "
        "and the encoder-vs-LLM comparison cannot be computed. Check that the "
        "stage2 predictions store is attached as a notebook input and that its "
        "'approach' column reads 'finetuned'.")
if _n_llm == 0:
    raise RuntimeError("head contains ZERO prompted LLM rows. Check stage3/3b inputs.")

KEYS = ["model", "model_tag", "tier", "task", "dataset", "eval_regime"]
rows = []
for kv, g in head.groupby(KEYS, dropna=False):
    rec = dict(zip(KEYS, kv))
    if rec["task"] not in LABELSETS:
        continue
    rec.update(cell(g, LABELSETS[rec["task"]]))
    rows.append(rec)
metrics = pd.DataFrame(rows)
metrics.to_csv(f"{OUT}/tab9_evaluability.csv", index=False)

n_drop = int((~metrics.reportable).sum())
print(f"[gate] {n_drop}/{len(metrics)} cells not reportable "
      f"-> excluded from tables and greyed in figures")
for r in metrics[~metrics.reportable].itertuples():
    print(f"        {r.model:26s} {r.task:13s} {r.dataset:8s} {r.eval_regime:14s} "
          f"n={r.n:<5d} {r.exclusion_reason}")

REP = metrics[metrics.reportable].copy()


# ============================================================================
# TABLE 0 - majority-class baselines
# ============================================================================
base_rows = []
for (task, ds), g in head.groupby(["task", "dataset"]):
    if task not in LABELSETS:
        continue
    gold = g.drop_duplicates("id")
    maj = gold.y_true.value_counts().idxmax()
    yp = np.full(len(gold), maj)
    base_rows.append({
        "task": task, "dataset": ds, "n": len(gold), "majority_class": maj,
        "macro_f1": round(float(f1_score(gold.y_true, yp, labels=LABELSETS[task],
                                         average="macro", zero_division=0)), 4),
        "accuracy": round(float(accuracy_score(gold.y_true, yp)), 4)})
baselines = pd.DataFrame(base_rows)
baselines.to_csv(f"{OUT}/tab0_baselines.csv", index=False)
BASE_F1 = {(r.task, r.dataset): r.macro_f1 for r in baselines.itertuples()}


# ============================================================================
# TABLES 1 & 2 - main results, STRATIFIED BY EVALUATION REGIME
#
# Averaging in-domain and transfer performance for the encoders, then comparing
# that mean against prompted LLMs, understates the encoders by construction and
# inverts the headline finding. The two regimes answer different questions and
# are reported as two separate panels.
# ============================================================================
IN_DOMAIN = ["in_domain", "prompted"]          # prompted LLMs have no training set
TRANSFER = ["cross_dataset", "cross_project", "prompted"]

# Short regime tags for column headers where one (task, dataset) cell holds
# more than one evaluation regime.
REG_TAG = {"cross_project": "x-proj", "cross_dataset": "x-data",
           "prompted": "zero-shot", "in_domain": "in-domain"}


def panel(regimes, label, value="macro_f1", split_regimes=False):
    d = REP[REP.eval_regime.isin(regimes)].copy()
    if split_regimes:
        d["col"] = (d.task.map(TASK_LABEL) + "\n" + d.dataset.str.upper()
                    + " / " + d.eval_regime.map(REG_TAG))
    else:
        d["col"] = d.task.map(TASK_LABEL) + "\n" + d.dataset.str.upper()
    # One value per (model, col) cell, asserted rather than assumed. The old
    # aggfunc="max" hid a real collision: with cross_project AND cross_dataset
    # both in `regimes` and no regime in the column key, an encoder's transfer
    # cell silently became max(cross_project, cross_dataset) - 0.8649 instead
    # of 0.5279 for BERT on security/PROMISE - under a caption that described
    # corpus transfer. The transfer panel now carries the regime in the column
    # header (split_regimes=True), so the two quantities are never blended,
    # and any future collision fails loudly here instead of silently.
    assert not d.duplicated(["model", "tier", "col"]).any(), (
        f"panel({label}): two rows share one cell; add the regime to the "
        f"column key (split_regimes=True)")
    piv = d.pivot_table(index=["model", "tier"], columns="col",
                        values=value, aggfunc="max")
    if split_regimes:
        wanted = [f"{TASK_LABEL[t]}\n{ds.upper()} / {REG_TAG[r]}"
                  for t in ["fr_nfr", "security", "subtype_top4", "subtype_top6", "subtype_all"]
                  for ds in ["promise", "secreq"]
                  for r in ["cross_project", "cross_dataset", "prompted"]]
    else:
        wanted = [f"{TASK_LABEL[t]}\n{ds.upper()}"
                  for t in ["fr_nfr", "security", "subtype_top4", "subtype_top6", "subtype_all"]
                  for ds in ["promise", "secreq"]]
    order = [c for c in wanted if c in piv.columns]
    piv = piv[order]
    piv = piv.reset_index()
    piv["_t"] = piv.tier.map({"finetuned": 0, "commercial": 1,
                              "open_hosted": 2, "open_local": 3}).fillna(9)
    piv = piv.sort_values(["_t", "model"]).drop(columns="_t")
    piv.attrs["label"] = label
    return piv


def to_latex(piv, caption, label, path):
    """Escapes every underscore and ampersand, fixes the column format, and
    emits booktabs rules. pandas' default output contains raw task ids like
    fr_nfr, which do not compile outside math mode."""
    df = piv.copy()
    df.columns = [str(c).replace("\n", " -- ") for c in df.columns]
    esc = lambda s: (str(s).replace("\\", r"\textbackslash{}")
                     .replace("_", r"\_").replace("&", r"\&").replace("%", r"\%"))
    num = [c for c in df.columns if c not in ("model", "tier")]
    best = {c: df[c].max() for c in num}
    lines = [r"\begin{table}[t]", r"\centering", r"\small",
             rf"\caption{{{caption}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{ll" + "r" * len(num) + "}", r"\toprule",
             " & ".join([r"\textbf{Model}", r"\textbf{Tier}"] +
                        [rf"\textbf{{{esc(c)}}}" for c in num]) + r" \\",
             r"\midrule"]
    prev = None
    for _, r in df.iterrows():
        if prev is not None and r["tier"] != prev:
            lines.append(r"\midrule")
        prev = r["tier"]
        cells = []
        for c in num:
            v = r[c]
            if pd.isna(v):
                cells.append("--")
            elif abs(v - best[c]) < 1e-9:
                cells.append(rf"\textbf{{{v:.3f}}}")
            else:
                cells.append(f"{v:.3f}")
        lines.append(" & ".join([esc(r["model"]), esc(r["tier"])] + cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    open(path, "w").write("\n".join(lines) + "\n")


p_in = panel(IN_DOMAIN, "in-domain")
p_tr = panel(TRANSFER, "transfer", split_regimes=True)
p_in.to_csv(f"{OUT}/tab1_main_in_domain.csv", index=False)
p_tr.to_csv(f"{OUT}/tab2_main_transfer.csv", index=False)
to_latex(p_in,
         "Macro-F1, in-domain evaluation. Encoders are fine-tuned and tested on "
         "the same corpus; LLMs are zero-shot and have no training distribution. "
         "Best per column in bold.",
         "tab:main-indomain", f"{OUT}/tab1_main_in_domain.tex")
to_latex(p_tr,
         "Macro-F1 under distribution shift, by transfer regime: x-proj = "
         "trained on the other projects of the same corpus (cross-project), "
         "x-data = trained on the other corpus (cross-dataset). Zero-shot "
         "columns repeat the prompted-LLM results for reference, because a "
         "prompted model has no training corpus to shift from. "
         "Best per column in bold.",
         "tab:main-transfer", f"{OUT}/tab2_main_transfer.tex")

# ------------------------------------------------------------------------ [FIX-2]
# RQ1 asks for macro- AND weighted-F1. Same panels, second metric.
panel(IN_DOMAIN, "in-domain", "weighted_f1").to_csv(
    f"{OUT}/tab1w_main_in_domain_weightedF1.csv", index=False)
panel(TRANSFER, "transfer", "weighted_f1", split_regimes=True).to_csv(
    f"{OUT}/tab2w_main_transfer_weightedF1.csv", index=False)

# Sanity check on [FIX-1]: with encoders present these two panels MUST differ.
# If they are identical the encoder rows have gone missing again.
if p_in.to_csv(index=False) == p_tr.to_csv(index=False):
    raise RuntimeError(
        "tab1 and tab2 are identical. That can only happen if no row with an "
        "in_domain / cross_project / cross_dataset regime survived the head "
        "filter - i.e. the encoders were dropped. See [FIX-1].")
print(f"[tab1/2] regime-stratified main tables written "
      f"({int((REP.tier == 'finetuned').sum())} encoder cells included)")


# ============================================================================
# TABLE 3 - generalisation gap (RQ2)
#
# Two distinct quantities, never merged into one column:
#   encoders: train_gap    = in-domain minus transferred (a training effect)
#   LLMs:     corpus_delta = corpus A minus corpus B (no training involved)
# ============================================================================
gap_rows = []
for (m, tier, task), g in REP.groupby(["model", "tier", "task"]):
    if tier == "finetuned":
        for ds in g.dataset.unique():
            ref = g[(g.dataset == ds) & (g.eval_regime == "in_domain")]
            if not len(ref):
                continue
            r0 = ref.macro_f1.iloc[0]
            # [FIX-3] one row per transfer regime. cross_project (unseen
            # projects, same corpus) and cross_dataset (unseen corpus, shifted
            # class prior) are different quantities and must not be collapsed
            # by a max().
            for reg in ["cross_project", "cross_dataset"]:
                tra = g[(g.dataset == ds) & (g.eval_regime == reg)]
                if not len(tra):
                    continue
                t0 = tra.macro_f1.max()
                gap_rows.append({
                    "model": m, "tier": tier, "task": task, "evaluated_on": ds,
                    "oos_regime": reg,
                    "measure": "train_gap (in-domain - transferred)",
                    "reference_f1": r0, "transfer_f1": t0,
                    "gap": round(r0 - t0, 4),
                    "relative_drop_pct": (round(100 * (r0 - t0) / r0, 1)
                                          if r0 >= 0.30 else np.nan)})
    else:
        d = g[g.eval_regime == "prompted"]
        if d.dataset.nunique() < 2:
            continue
        a = d[d.dataset == "promise"].macro_f1.max()
        b = d[d.dataset == "secreq"].macro_f1.max()
        if np.isnan(a) or np.isnan(b):
            continue
        gap_rows.append({
            "model": m, "tier": tier, "task": task, "evaluated_on": "promise vs secreq",
            "oos_regime": "n/a (zero-shot)",
            "measure": "corpus_delta (no training distribution)",
            "reference_f1": a, "transfer_f1": b, "gap": round(a - b, 4),
            "relative_drop_pct": round(100 * (a - b) / a, 1) if a >= 0.30 else np.nan})
# A declared column list, so an empty result is a readable file with a header
# rather than a KeyError. pd.DataFrame([]).sort_values(["task", ...]) raises,
# and the `if len(gaps)` guard further down never gets the chance to run.
GAP_COLS = ["model", "tier", "task", "evaluated_on", "oos_regime", "measure",
            "reference_f1", "transfer_f1", "gap", "relative_drop_pct"]
gaps = pd.DataFrame(gap_rows, columns=None if gap_rows else GAP_COLS)
if len(gaps):
    gaps = gaps.sort_values(["task", "tier", "gap"])
gaps.to_csv(f"{OUT}/tab3_generalisation_gap.csv", index=False)
print(f"[tab3] {len(gaps)} gap rows")


# ============================================================================
# TABLE 4 - McNemar, paired on identical items, against EVERY encoder regime
#
# Testing LLMs only against the transferred encoder silently compares them with
# the weakest available baseline. Both regimes are tested; the in-domain arm is
# the one the headline claim rests on.
# ============================================================================
def mcnemar(a_ok, b_ok):
    b = int(np.sum(a_ok & ~b_ok))
    c = int(np.sum(~a_ok & b_ok))
    if b + c == 0:
        return b, c, 1.0
    return b, c, float(sps.binomtest(min(b, c), b + c, 0.5).pvalue)


def multiplicity(p_values):
    """Holm and Benjamini-Hochberg adjusted p-values for one family of tests.

    Pulled out of tab4 so that every family gets the same treatment. It used to
    be inline there, which is how tab7 ended up reporting 90 McNemar tests at a
    raw alpha of 0.05 while tab4 corrected its own 186 and Stage 3b-repair
    corrected its few-shot family with Holm. Three families, three conventions,
    in one study - and tab7's uncorrected count was the one that made few-shot
    look like it helped."""
    p = np.asarray(p_values, dtype=float)
    n = len(p)
    if n == 0:
        return p, p
    o = np.argsort(p)
    holm = np.empty(n)
    holm[o] = np.minimum(np.maximum.accumulate(p[o] * (n - np.arange(n))), 1.0)
    bh = np.empty(n)
    bh[o] = np.minimum(
        np.minimum.accumulate((p[o] * n / (np.arange(n) + 1))[::-1])[::-1], 1.0)
    return holm, bh


mc_rows = []
for (task, ds), g in head.groupby(["task", "dataset"]):
    if task not in LABELSETS:
        continue
    keep = set(zip(REP.model, REP.task, REP.dataset, REP.eval_regime))
    # cross_project_lopo is a SECOND partition of the cross-project experiment
    # over the same 968 requirements (it exists for tab11's per-project view),
    # and its predictions agree with the grouped-CV arm on ~94% of items. Both
    # entering this family made 16 of the 276 tests near-duplicates of tests
    # already present, which inflates the family size the BH/Holm correction
    # divides by and tests one hypothesis twice. One partition per encoder.
    fts = [(m, r) for m, r in
           g[g.tier == "finetuned"][["model", "eval_regime"]].drop_duplicates().values
           if (m, task, ds, r) in keep and r != "cross_project_lopo"]
    llms = [m for m in g[g.tier != "finetuned"].model.unique()
            if (m, task, ds, "prompted") in keep]
    for ftm, reg in fts:
        A = g[(g.model == ftm) & (g.eval_regime == reg)].drop_duplicates("id").set_index("id")
        for lm in llms:
            B = g[(g.model == lm) & (g.eval_regime == "prompted")].drop_duplicates("id").set_index("id")
            ids = sorted(set(A.index) & set(B.index))
            if len(ids) < MIN_N:
                continue
            a_ok = (A.loc[ids].y_hat.values == A.loc[ids].y_true.values)
            b_ok = (B.loc[ids].y_hat.values == B.loc[ids].y_true.values)
            b, c, p = mcnemar(a_ok, b_ok)
            mc_rows.append({
                "task": task, "dataset": ds, "encoder": ftm, "encoder_regime": reg,
                "llm": lm, "n_paired": len(ids),
                "encoder_acc": round(float(a_ok.mean()), 4),
                "llm_acc": round(float(b_ok.mean()), 4),
                "encoder_only_correct": b, "llm_only_correct": c,
                "llm_advantage": c - b, "p_value": p})
# [FIX-4] fixed schema, so an empty result is still a readable file rather than
# a one-byte mystery, and the run fails loudly instead of silently.
MCN_COLS = ["task", "dataset", "encoder", "encoder_regime", "llm", "n_paired",
            "encoder_acc", "llm_acc", "encoder_only_correct", "llm_only_correct",
            "llm_advantage", "p_value", "p_holm", "p_bh",
            "sig_raw", "sig_bh", "sig_holm"]
mcn = pd.DataFrame(mc_rows, columns=None if mc_rows else MCN_COLS)

if not len(mcn):
    # Write the empty table WITH its header BEFORE failing. MCN_COLS was
    # declared for exactly this case and was unreachable: the raise came first,
    # so the run left a traceback and no file at all.
    mcn.to_csv(f"{OUT}/tab4_mcnemar.csv", index=False)
    raise RuntimeError(
        "tab4: zero McNemar tests were produced. Every encoder-vs-LLM "
        "comparison would then be reported without a significance test. "
        "This is the symptom of the [FIX-1] defect - check the head filter. "
        "An empty tab4_mcnemar.csv with headers has been written.")

if len(mcn):
    n = len(mcn)
    holm, bh = multiplicity(mcn.p_value.values)
    mcn["p_holm"] = holm.round(5)
    mcn["p_bh"] = bh.round(5)
    mcn["sig_raw"] = mcn.p_value < ALPHA
    mcn["sig_bh"] = mcn.p_bh < ALPHA
    mcn["sig_holm"] = mcn.p_holm < ALPHA
    mcn["p_value"] = mcn.p_value.round(6)
    print(f"[tab4] {n} tests | sig raw {int(mcn.sig_raw.sum())} | "
          f"BH {int(mcn.sig_bh.sum())} | Holm {int(mcn.sig_holm.sum())}")
    mcn["verdict"] = np.where(
        mcn.p_bh >= ALPHA, "n.s.",
        np.where(mcn.llm_acc > mcn.encoder_acc, "LLM wins", "Encoder wins"))
mcn.to_csv(f"{OUT}/tab4_mcnemar.csv", index=False)


# ============================================================================
# TABLE 5 - sub-type comparison, like-for-like on shared items
# ============================================================================
sub_rows = []
for task in SUBTYPE_TASKS:
    # IN_DOMAIN, not every regime. This table sets fine-tuned encoders against
    # PROMPTED LLMs, and a prompted model has no training distribution, so the
    # encoder's side of that comparison is its in-domain number - the same one
    # tab1 and fig4 show. Until Stage 1 gained the cross-project sub-type
    # families the encoders had only in-domain rows here and no filter was
    # needed; with both regimes present an unfiltered groupby("model") pooled
    # them, reporting n_own = 1048 for a 524-item corpus and a macro-F1 that
    # was the mean of two different experiments (0.7064 where the in-domain
    # value is 0.7240). The transfer arms are reported in tab2 and tab3.
    g = head[(head.task == task) & (head.eval_regime.isin(IN_DOMAIN))]
    if not len(g):
        continue
    ls = LABELSETS[task]
    rep_models = set(REP[(REP.task == task)].model)
    g = g[g.model.isin(rep_models)]
    if not len(g):
        continue
    # Restrict the shared set to CORE wherever the store marks it. Without this,
    # "common" is whatever happens to overlap between a local model scored on
    # the full frame and an API model capped at 60 items by its free quota -
    # which is the comparison stage3b-repair added `in_core` to prevent.
    gsub = g[g.in_core] if g.in_core.any() else g
    sets = {m: set(x.id) for m, x in gsub.groupby("model")}
    common = set.intersection(*sets.values()) if sets else set()
    ok_common, _, _ = evaluability(
        gsub[gsub.id.isin(common)].drop_duplicates("id").y_true, ls) if common else (False, 0, {})
    for m, x in g.groupby("model"):
        tier = x.tier.iloc[0]
        own = f1_score(x.y_true, x.y_hat, labels=ls, average="macro", zero_division=0)
        cc = x[x.id.isin(common)]
        cf = (f1_score(cc.y_true, cc.y_hat, labels=ls, average="macro", zero_division=0)
              if len(cc) >= MIN_N else np.nan)
        sub_rows.append({
            "task": task, "model": m, "tier": tier,
            "n_own": len(x), "macro_f1_own": round(float(own), 4),
            "n_common": len(cc),
            "macro_f1_common": round(float(cf), 4) if not np.isnan(cf) else np.nan,
            "common_set_reportable": bool(ok_common)})
# Same guard as tab3: with no sub-type store attached, sub_rows is empty and
# the unguarded sort_values() took the whole notebook down with a KeyError.
SUB_COLS = ["task", "model", "tier", "n_own", "macro_f1_own", "n_common",
            "macro_f1_common", "common_set_reportable"]
subtab = pd.DataFrame(sub_rows, columns=None if sub_rows else SUB_COLS)
if len(subtab):
    subtab = subtab.sort_values(["task", "macro_f1_own"], ascending=[True, False])
subtab.to_csv(f"{OUT}/tab5_subtype.csv", index=False)


# ============================================================================
# TABLE 10 - like-for-like ranking on the shared subset
#
# tab1 and tab2 report each model on everything it answered, and that is NOT the
# same item set across tiers: a local model runs the full frame while an API
# model is capped by its provider's free daily quota (Gemini answered 60 of 300
# on the committed run). Per-model that is the honest number; as a RANKING it
# compares different exams. This table scores every model on the exact
# intersection and states the intersection's size and rarest-class support, so
# the comparison can be shown to be valid rather than asserted.
# ============================================================================
lfl = []
# Same restriction as tab5, and for the same reason: this is an encoder-vs-LLM
# ranking, so the encoder enters on its in-domain arm. Without it the loop below
# relied on drop_duplicates("id") to pick one of the two encoder regimes, which
# silently returns whichever Stage 2 happened to write first - a number decided
# by the order Stage 1 lists its families, not by any choice made here.
_lfl_head = head[head.eval_regime.isin(IN_DOMAIN)]
for (task, ds), g in _lfl_head.groupby(["task", "dataset"]):
    if task not in LABELSETS:
        continue
    ls = LABELSETS[task]
    gsub = g[g.in_core] if g.in_core.any() else g
    # Only models that already passed the evaluability gate take part. A single
    # badly rate-limited model would otherwise decide the intersection for
    # everyone: on the committed run the OpenRouter route answered 21 of 300
    # items, which collapsed every binary comparison below MIN_N and produced a
    # table with no comparison in it at all. Whatever is dropped is NAMED, so
    # the restriction is visible rather than a silent cap.
    rep_models = set(REP[(REP.task == task) & (REP.dataset == ds)].model)
    # Computed from g, not gsub: anything the in_core filter above removed must
    # still be NAMED. Taking it from gsub meant a model dropped by that filter
    # was not even listed as excluded, and the column read "-".
    # Two ways to leave this table, both named: failing the evaluability gate,
    # and passing it but having no rows inside CORE (a gate-passing model
    # dropped by the in_core filter is absent from `sets` and would otherwise
    # vanish silently, since being in rep_models removes it from the first
    # subtraction).
    excluded = sorted(set(g.model.unique()) - rep_models)
    excluded += sorted(rep_models - set(gsub.model.unique()))
    gsub = gsub[gsub.model.isin(rep_models)]
    sets = {m: set(x.id) for m, x in gsub.groupby("model")}
    if len(sets) < 2:
        continue
    common = set.intersection(*sets.values())
    if len(common) < MIN_N:
        lfl.append({"task": task, "dataset": ds, "model": "(no comparison)",
                    "tier": "-", "n_common": len(common),
                    "macro_f1_common": np.nan, "intersection_reportable": False,
                    "intersection_min_support": 0,
                    "excluded_models": ";".join(excluded) or "-",
                    "note": f"intersection below MIN_N={MIN_N}"})
        continue
    gold = gsub[gsub.id.isin(common)].drop_duplicates("id")
    ok_c, min_sup, _bad = evaluability(gold.y_true, ls)
    for m, x in gsub.groupby("model"):
        cc = x[x.id.isin(common)].drop_duplicates("id")
        lfl.append({
            "task": task, "dataset": ds, "model": m, "tier": cc.tier.iloc[0],
            "n_common": len(cc),
            "macro_f1_common": round(float(f1_score(
                cc.y_true, cc.y_hat, labels=ls, average="macro",
                zero_division=0)), 4),
            "intersection_reportable": bool(ok_c),
            "intersection_min_support": min_sup,
            "excluded_models": ";".join(excluded) or "-",
            "note": "-" if ok_c else "rarest class below MIN_CLASS_SUPPORT"})
likeforlike = pd.DataFrame(lfl)
if len(likeforlike):
    likeforlike = likeforlike.sort_values(
        ["task", "dataset", "macro_f1_common"], ascending=[True, True, False])
likeforlike.to_csv(f"{OUT}/tab10_like_for_like.csv", index=False)
print(f"[tab10] {len(likeforlike)} row(s) scored on the shared subset")


# ============================================================================
# TABLE 6 - per-class precision / recall / F1 with support
# ============================================================================
pc = []
for (m, tier, task, ds, reg), g in head.groupby(
        ["model", "tier", "task", "dataset", "eval_regime"]):
    if task not in LABELSETS:
        continue
    ls = LABELSETS[task]
    pr, rc, f1v, sup = precision_recall_fscore_support(
        g.y_true, g.y_hat, labels=ls, zero_division=0)
    for c, a, b, f, s in zip(ls, pr, rc, f1v, sup):
        pc.append({"model": m, "tier": tier, "task": task, "dataset": ds,
                   "eval_regime": reg, "category": c,
                   "precision": round(float(a), 4), "recall": round(float(b), 4),
                   "f1": round(float(f), 4), "support": int(s),
                   "reliable": bool(s >= MIN_CLASS_SUPPORT)})
perclass = pd.DataFrame(pc)
perclass.to_csv(f"{OUT}/tab6_perclass.csv", index=False)


# ============================================================================
# TABLE 7 - few-shot, paired on identical items
# ============================================================================
fs = []
bp = store[store.prompt_id == "base"]
for (m, task, ds), g in bp.groupby(["model", "task", "dataset"]):
    if task not in LABELSETS:
        continue
    ks = {int(k): x for k, x in g.groupby("shot_k")}
    if 0 not in ks or len(ks) < 2:
        continue
    ls = LABELSETS[task]
    shots = sorted(k for k in ks if k > 0)
    common = set.intersection(*[set(ks[k].id) for k in [0] + shots])
    if len(common) < MIN_N:
        continue
    z = ks[0][ks[0].id.isin(common)].drop_duplicates("id").sort_values("id")
    f0 = f1_score(z.y_true, z.y_hat, labels=ls, average="macro", zero_division=0)
    for k in shots:
        s = ks[k][ks[k].id.isin(common)].drop_duplicates("id").sort_values("id")
        fk = f1_score(s.y_true, s.y_hat, labels=ls, average="macro", zero_division=0)
        b, c, p = mcnemar((z.y_hat.values == z.y_true.values),
                          (s.y_hat.values == s.y_true.values))
        fs.append({"model": m, "task": task, "dataset": ds, "k": k,
                   "n_paired": len(common), "k0_f1": round(float(f0), 4),
                   "k_f1": round(float(fk), 4), "gain": round(float(fk - f0), 4),
                   "mcnemar_p": round(p, 5), "sig_raw": bool(p < ALPHA)})
fewshot = pd.DataFrame(fs)
if len(fewshot):
    # Same family, same correction as tab4. `sig` now keys on BH, so a reader
    # comparing tab7 with tab4 or with s3b_fix_fewshot_paired.csv is comparing
    # like with like instead of a raw alpha against two corrected ones.
    _holm, _bh = multiplicity(fewshot.mcnemar_p.values)
    fewshot["p_holm"] = _holm.round(5)
    fewshot["p_bh"] = _bh.round(5)
    fewshot["sig_bh"] = fewshot.p_bh < ALPHA
    fewshot["sig_holm"] = fewshot.p_holm < ALPHA
    fewshot["sig"] = fewshot.sig_bh
    print(f"[tab7] {len(fewshot)} few-shot tests | sig raw "
          f"{int(fewshot.sig_raw.sum())} | BH {int(fewshot.sig_bh.sum())} | "
          f"Holm {int(fewshot.sig_holm.sum())}")
fewshot.to_csv(f"{OUT}/tab7_fewshot.csv", index=False)


# ============================================================================
# TABLE 8 - prompt sensitivity
# ============================================================================
ps = []
for (m, task, ds), g in store[store.shot_k == 0].groupby(["model", "task", "dataset"]):
    if task not in LABELSETS or g.prompt_id.nunique() < 2:
        continue
    ls = LABELSETS[task]
    ids = set.intersection(*[set(x.id) for _, x in g.groupby("prompt_id")])
    if len(ids) < MIN_N:
        continue
    rec = {"model": m, "task": task, "dataset": ds, "n_paired": len(ids)}
    for pid, x in g.groupby("prompt_id"):
        y = x[x.id.isin(ids)].drop_duplicates("id")
        rec[f"f1_{pid}"] = round(float(f1_score(y.y_true, y.y_hat, labels=ls,
                                                average="macro", zero_division=0)), 4)
    vals = [v for k, v in rec.items() if k.startswith("f1_")]
    rec["spread"] = round(max(vals) - min(vals), 4)
    ps.append(rec)
prompts = pd.DataFrame(ps)
prompts.to_csv(f"{OUT}/tab8_prompt_sensitivity.csv", index=False)


# ============================================================================
# TABLE 11 - per-project performance, encoder AND prompted LLM
#
# RQ2 asks two things: how much performance drops out of distribution, and
# whether the drop DIFFERS BY MODEL CLASS. The pooled cross-project number
# answers the first and is silent on the second, because it averages 47 projects
# into one figure. This table keeps them apart.
#
# It reports ACCURACY per project, not macro-F1. On PROMISE, 24 of the 47
# single-project test sets contain only one of the two FR/NFR classes, and a
# macro-F1 over a declared label set scores the absent class 0 and still divides
# by K - a downward bias that is an artefact of the fold size. Accuracy is
# well defined on every fold. macro-F1 is reported too, but only for the folds
# where every declared class is actually present, and the count of those folds
# is stated so the two columns are never confused.
# ============================================================================
UNI_PATH = find("/kaggle/input/**/unified.parquet", "/kaggle/input/**/unified.csv")
proj_rows = []
if UNI_PATH is None:
    print("[tab11] unified corpus not attached - per-project table skipped. "
          "Add the Stage 1 output as an input to enable it.")
else:
    _uni = read_any(UNI_PATH)
    _pmap = _uni.set_index(_uni["id"].astype(str))["project"].astype(str)
    _smap = _uni.set_index(_uni["id"].astype(str))["source_dataset"].astype(str)
    head = head.copy()
    head["project"] = head["id"].astype(str).map(_pmap)
    head["corpus"] = head["id"].astype(str).map(_smap)

    # Encoders are read from their cross-project folds; a prompted model has no
    # folds, so its rows are grouped by the project of the item it scored. Both
    # end up describing the same question: how does this model do on THIS
    # project's requirements?
    # Stage 1 can emit TWO cross-project partitions of one task: a 47-fold
    # leave-one-project-out family (regime "cross_project_lopo", built for this
    # table) and a 5-fold grouped-CV family (regime "cross_project", built for
    # the headline gap). They cover the SAME requirements, so taking both would
    # put every item in this table twice, and the drop_duplicates("id") below
    # would then keep whichever the store happened to write first. Where the
    # fine-grained family exists it is the right one here; where it does not,
    # the grouped-CV family is used.
    _cp = head[head.eval_regime.astype(str).str.startswith("cross_project")]
    _fine = set(map(tuple, _cp[_cp.eval_regime == "cross_project_lopo"]
                    [["model", "task", "corpus"]].drop_duplicates().values))
    if _fine:
        _cp = _cp[[(mm, tt, cc) not in _fine or rr == "cross_project_lopo"
                   for mm, tt, cc, rr in zip(_cp.model, _cp.task, _cp.corpus,
                                             _cp.eval_regime)]]
    pool = pd.concat([_cp, head[head.eval_regime == "prompted"]],
                     ignore_index=True)
    for (m, tier, task, corpus, project), g in pool.groupby(
            ["model", "tier", "task", "corpus", "project"], dropna=True):
        if task not in LABELSETS or not isinstance(project, str) or project == "nan":
            continue
        g = g.drop_duplicates("id")
        if len(g) < 5:
            continue
        ls = LABELSETS[task]
        present = set(g.y_true.astype(str))
        complete = set(ls) <= present
        proj_rows.append({
            "model": m, "tier": tier, "task": task, "corpus": corpus,
            "project": project, "n": len(g),
            "accuracy": round(float(accuracy_score(g.y_true, g.y_hat)), 4),
            "macro_f1": (round(float(f1_score(g.y_true, g.y_hat, labels=ls,
                                              average="macro", zero_division=0)), 4)
                         if complete else np.nan),
            "all_classes_present": bool(complete),
            "n_classes_present": len(present & set(ls)),
        })

perproject = pd.DataFrame(proj_rows)
if len(perproject):
    perproject = perproject.sort_values(["task", "corpus", "model", "accuracy"])
perproject.to_csv(f"{OUT}/tab11_per_project.csv", index=False)

# Spread summary: this is the sentence RQ2 wants, per model class.
spread_rows = []
if len(perproject):
    for (m, tier, task, corpus), g in perproject.groupby(
            ["model", "tier", "task", "corpus"]):
        if len(g) < 3:
            continue
        spread_rows.append({
            "model": m, "tier": tier, "task": task, "corpus": corpus,
            "n_projects": len(g),
            "acc_mean": round(float(g.accuracy.mean()), 4),
            "acc_median": round(float(g.accuracy.median()), 4),
            "acc_min": round(float(g.accuracy.min()), 4),
            "acc_max": round(float(g.accuracy.max()), 4),
            "acc_std": round(float(g.accuracy.std(ddof=1)), 4),
            "acc_iqr": round(float(g.accuracy.quantile(0.75) -
                                   g.accuracy.quantile(0.25)), 4),
            "worst_project": g.loc[g.accuracy.idxmin(), "project"],
            "n_items": int(g.n.sum()),
            "folds_with_all_classes": int(g.all_classes_present.sum()),
        })
spread = pd.DataFrame(spread_rows)
if len(spread):
    # A spread computed over 4 projects is not comparable with one computed over
    # 37, and the difference is entirely an artefact of API quota: a model capped
    # at 60 core items simply touches fewer projects. Reading its narrower sd as
    # "more stable" would invert the finding. Flag it rather than trusting the
    # reader to check n_projects.
    # Counting PROJECTS alone was not enough. SecReq has only three projects, so
    # a model capped at 60 items still touches all three and scored 100%
    # coverage - certified comparable while resting on a fraction of the data,
    # which is precisely the case this flag exists to catch. Coverage is now the
    # weaker of the two ratios, and the item total is emitted so the reader can
    # see what the flag is based on.
    _maxp = spread.groupby(["task", "corpus"]).n_projects.transform("max")
    _maxi = spread.groupby(["task", "corpus"]).n_items.transform("max")
    spread["project_coverage_pct"] = (100 * spread.n_projects / _maxp).round(1)
    spread["item_coverage_pct"] = (100 * spread.n_items / _maxi).round(1)
    spread["coverage_pct"] = spread[["project_coverage_pct",
                                     "item_coverage_pct"]].min(axis=1).round(1)
    spread["spread_comparable"] = spread.coverage_pct >= 80.0
    spread = spread.sort_values(["task", "corpus", "spread_comparable", "acc_std"],
                                ascending=[True, True, False, False])
    _n_bad = int((~spread.spread_comparable).sum())
    if _n_bad:
        print(f"        NOTE: {_n_bad} row(s) cover <80% of the projects OR of "
              f"the items the best-covered model saw; their spread is NOT "
              f"comparable and is flagged spread_comparable=False.")
spread.to_csv(f"{OUT}/tab11b_per_project_spread.csv", index=False)
print(f"[tab11] {len(perproject)} model x project cell(s); "
      f"{len(spread)} spread row(s)")
if len(spread):
    print("        widest per-project spread (the RQ2 'does it differ by model "
          "class?' evidence):")
    for r in spread.head(5).itertuples():
        print(f"          {r.model:26s} {r.task:13s} {r.corpus:8s} "
              f"acc {r.acc_min:.3f}-{r.acc_max:.3f} "
              f"(median {r.acc_median:.3f}, sd {r.acc_std:.3f}, "
              f"worst: project {r.worst_project})")


# ============================================================================
# FIGURES
# ============================================================================
plt.rcParams.update({"font.size": 11, "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 160,
                     "savefig.bbox": "tight", "axes.grid": True,
                     "grid.alpha": 0.25, "grid.linestyle": ":"})


def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(f"{OUT}/{name}.{ext}")
    plt.close(fig)
    print(f"[fig] {name}")


# ------------------------------------------------------------------------ [FIX-7]
# With the encoders restored, each panel carries four more rows and the model
# labels grew ("BERT (weighted)  (n=968)"). At the old spacing matplotlib drew
# those labels straight over the bars of the panel to their left. Every
# multi-panel figure now reserves horizontal room between panels.
PANEL_W = 7.4          # inches per panel, was 6.2
PANEL_WSPACE = 0.62    # fraction of panel width kept clear for tick labels


def legend(fig, tiers, extra=None):
    h = [Line2D([0], [0], marker="s", ls="", ms=10, color=TIER_COLOR[t],
                label=TIER_LABEL[t]) for t in tiers if t in TIER_COLOR]
    if extra:
        h += extra
    fig.legend(handles=h, loc="lower center", ncol=min(5, len(h)),
               frameon=False, bbox_to_anchor=(0.5, -0.06))


def barh_panel(ax, d, title, base_line=None, ref_line=None, ref_label=""):
    d = d.sort_values("macro_f1")
    y = np.arange(len(d))
    err = np.vstack([(d.macro_f1 - d.ci_lo).clip(lower=0),
                     (d.ci_hi - d.macro_f1).clip(lower=0)])
    ax.barh(y, d.macro_f1, color=[TIER_COLOR.get(t, "#888") for t in d.tier],
            height=0.68, xerr=err, error_kw=dict(ecolor="#374151", lw=1.1, capsize=2.5))
    ax.set_yticks(y)
    ax.set_yticklabels([f"{m}  (n={n})" for m, n in zip(d.model, d.n)], fontsize=9)
    for i, v in enumerate(d.macro_f1):
        ax.text(v + 0.012, i, f"{v:.3f}", va="center", fontsize=8.5, weight="bold")
    if base_line is not None:
        ax.axvline(base_line, color=TIER_COLOR["baseline"], ls="-", lw=1.6, zorder=0)
    if ref_line is not None:
        ax.axvline(ref_line, color=TIER_COLOR["finetuned"], ls="--", lw=1.6, zorder=0)
    ax.set_xlim(0, 1.18)
    ax.set_xticks(np.arange(0, 1.01, 0.25))
    ax.set_xlabel("macro-F1")
    ax.set_title(title, fontsize=11, weight="bold")


# --- fig1: RQ1 in-domain, encoders included -----------------------------------
cells1 = [("fr_nfr", "promise"), ("security", "promise"), ("security", "secreq")]
cells1 = [c for c in cells1 if len(REP[(REP.task == c[0]) & (REP.dataset == c[1])])]
if cells1:
    fig, axes = plt.subplots(1, len(cells1), figsize=(PANEL_W * len(cells1), 6),
                             gridspec_kw={"wspace": PANEL_WSPACE})
    axes = np.atleast_1d(axes)
    for ax, (t, ds) in zip(axes, cells1):
        d = REP[(REP.task == t) & (REP.dataset == ds) &
                (REP.eval_regime.isin(IN_DOMAIN))]
        barh_panel(ax, d, f"{TASK_LABEL[t]} - {ds.upper()}",
                   base_line=BASE_F1.get((t, ds)))
    fig.suptitle("RQ1 - In-domain: fine-tuned encoders vs zero-shot LLMs",
                 fontsize=14, weight="bold", y=1.01)
    fig.text(0.5, 0.965, "Grey line = majority-class baseline. "
             "Error bars = 95% stratified bootstrap CI. Cells failing the "
             "evaluability gate are omitted.", ha="center", fontsize=9.5, color="#4B5563")
    legend(fig, ["finetuned", "commercial", "open_hosted", "open_local"],
           [Line2D([0], [0], color=TIER_COLOR["baseline"], lw=2, label="Majority baseline")])
    save(fig, "fig1_rq1_in_domain")

# --- fig2: RQ2 transfer -------------------------------------------------------
cells2 = [c for c in [("security", "secreq"), ("security", "promise")]
          if len(REP[(REP.task == c[0]) & (REP.dataset == c[1])])]
if cells2:
    fig, axes = plt.subplots(1, len(cells2), figsize=(PANEL_W * len(cells2), 6),
                             gridspec_kw={"wspace": PANEL_WSPACE})
    axes = np.atleast_1d(axes)
    for ax, (t, ds) in zip(axes, cells2):
        # ONE encoder regime per panel. TRANSFER holds cross_dataset AND
        # cross_project, and barh_panel labels a bar "<model>  (n=<n>)" with
        # no regime - so an unfiltered panel drew "BERT (weighted)  (n=968)"
        # twice, at 0.865 (trained on PROMISE itself) and 0.528 (trained on
        # SecReq), with nothing to tell them apart and a caption describing
        # only the second. This figure makes the CORPUS-shift claim, so the
        # encoders enter on their cross_dataset arm; the cross-project arm is
        # in tab2 and fig3.
        d = REP[(REP.task == t) & (REP.dataset == ds) &
                (REP.eval_regime.isin(["cross_dataset", "prompted"]))]
        ftbest = d[d.tier == "finetuned"].macro_f1.max() if (d.tier == "finetuned").any() else None
        barh_panel(ax, d, f"Evaluated on {ds.upper()}",
                   base_line=BASE_F1.get((t, ds)), ref_line=ftbest)
    fig.suptitle("RQ2 - Under corpus shift: transferred encoders vs zero-shot LLMs",
                 fontsize=14, weight="bold", y=1.01)
    fig.text(0.5, 0.965, "Teal dashed line = best transferred encoder. Encoders "
             "were TRAINED ON THE OTHER CORPUS (cross-dataset); LLMs saw "
             "neither. Cross-project encoder results are in Table 2 and Figure 3.",
             ha="center", fontsize=9.5, color="#4B5563")
    legend(fig, ["finetuned", "commercial", "open_hosted", "open_local"])
    save(fig, "fig2_rq2_transfer")

# --- fig3: generalisation gap -------------------------------------------------
# [FIX-10] Plot EVERY gap row, not only the security task.
# The two cross-project rows (FR/NFR: BERT 0.065, RoBERTa 0.043 - the earlier
# 0.053/0.036 quoted here were the LOPO partition's gaps, which tab3 does not
# report) were being filtered out, yet they carry the central argument: an encoder facing
# UNSEEN PROJECTS loses 4-6%, while the same encoder facing an UNSEEN CORPUS
# with a shifted class prior loses 17-43%. Dropping them left the reader unable
# to see that the collapse is caused by prior shift and not by unfamiliar
# wording. The regime now appears in every label so the two are never conflated.
if len(gaps):
    d = gaps.copy()
    d["kind"] = np.where(d.tier == "finetuned", "encoder train_gap", "LLM corpus_delta")
    REGIME_TAG = {"cross_project": "cross-project",
                  "cross_dataset": "cross-dataset",
                  "n/a (zero-shot)": "corpus delta"}
    # The TASK is part of the identity of a gap row: tab3 spans five tasks, so
    # a label built from model/corpus/regime alone repeated "BERT (weighted)
    # ->PROMISE [cross-project]" five times, twice at values (0.0784, 0.0779)
    # no reader could tell apart.
    d["lab"] = [
        (f"{m} \u2192{ev.upper()}  {TASK_LABEL.get(tk, tk)}  [{REGIME_TAG.get(r, r)}]"
         if t == "finetuned"
         else f"{m}  {TASK_LABEL.get(tk, tk)}  [{REGIME_TAG.get(r, r)}]")
        for m, t, ev, r, tk in zip(d.model, d.tier, d.evaluated_on, d.oos_regime,
                                   d.task)]
    d = d.sort_values("gap")
    if len(d):
        fig, ax = plt.subplots(figsize=(11.5, max(4, 0.44 * len(d))))
        y = np.arange(len(d))
        ax.barh(y, d.gap, color=[TIER_COLOR.get(t, "#888") for t in d.tier], height=0.66)
        ax.set_yticks(y)
        ax.set_yticklabels(d.lab, fontsize=9)
        # headroom so the value labels never touch the axis or the tick text
        span = float(d.gap.max() - min(0.0, d.gap.min()))
        ax.set_xlim(min(0.0, d.gap.min()) - 0.10 * span, d.gap.max() + 0.10 * span)
        for i, v in enumerate(d.gap):
            off = 0.012 * span
            ax.text(v + (off if v >= 0 else -off), i, f"{v:+.3f}", va="center",
                    ha="left" if v >= 0 else "right", fontsize=8.5)
        ax.axvline(0, color="#111827", lw=1.2)
        ax.set_xlabel("macro-F1 lost (positive = performance drops off the source corpus)")
        ax.set_title("RQ2 - Generalisation gap: cross-project vs cross-dataset",
                     fontsize=13, weight="bold")
        fig.text(0.5, -0.02,
                 "Encoders: in-domain minus transferred (a training effect). LLMs: "
                 "PROMISE minus SecReq (no training distribution exists). The two "
                 "are different quantities and are labelled as such.\nNote the "
                 "contrast within the encoders: cross-project loss is of the same "
                 "order as the LLMs' corpus delta, while cross-dataset loss is "
                 "several times larger.",
                 ha="center", fontsize=9, color="#4B5563")
        save(fig, "fig3_generalisation_gap")

# --- fig4: sub-types ----------------------------------------------------------
st = [t for t in SUBTYPE_TASKS if len(REP[REP.task == t])]
if st:
    fig, axes = plt.subplots(1, len(st), figsize=(PANEL_W * len(st), 6),
                             gridspec_kw={"wspace": PANEL_WSPACE})
    axes = np.atleast_1d(axes)
    for ax, t in zip(axes, st):
        # The regime filter fig1 has always had, which this panel never needed
        # until the cross-project sub-type families existed. Without it each
        # encoder appears TWICE - once in-domain, once cross-project - with
        # nothing distinguishing the bars, under a title that says "in-domain".
        d = REP[(REP.task == t) & (REP.eval_regime.isin(IN_DOMAIN))]
        ftbest = d[d.tier == "finetuned"].macro_f1.max() if (d.tier == "finetuned").any() else None
        barh_panel(ax, d, TASK_LABEL[t], base_line=BASE_F1.get((t, "promise")),
                   ref_line=ftbest)
    fig.suptitle("NFR sub-type classification (in-domain)", fontsize=14,
                 weight="bold", y=1.01)
    fig.text(0.5, 0.965, "Teal dashed line = best fine-tuned encoder. "
             "Models whose rarest class fell below "
             f"{MIN_CLASS_SUPPORT} test items are not shown.",
             ha="center", fontsize=9.5, color="#4B5563")
    legend(fig, ["finetuned", "commercial", "open_hosted", "open_local"])
    save(fig, "fig4_subtypes")

# --- fig5: few-shot -----------------------------------------------------------
if len(fewshot):
    tasks = sorted(fewshot.task.unique())
    fig, axes = plt.subplots(1, len(tasks), figsize=(5.4 * len(tasks), 4.6),
                             squeeze=False, gridspec_kw={"wspace": 0.34})
    for ax, t in zip(axes[0], tasks):
        d = fewshot[fewshot.task == t]
        for m, g in d.groupby("model"):
            g = g.sort_values("k")
            ax.plot([0] + g.k.tolist(), [g.k0_f1.iloc[0]] + g.k_f1.tolist(),
                    marker="o", ms=4.5, lw=1.6, label=m)
        ax.set_title(TASK_LABEL.get(t, t), fontsize=11, weight="bold")
        # k means different things in the two harnesses: the binary tasks
        # count TOTAL exemplars in the prompt (k=2 is one per class), the
        # sub-type tasks count exemplars PER CLASS. One shared label would
        # misdescribe half the panels.
        ax.set_xlabel("shots per class (k)" if str(t).startswith("subtype")
                      else "exemplars in prompt, total (k)")
        ax.set_ylabel("macro-F1")
    axes[0][-1].legend(fontsize=7.5, frameon=False, loc="best", ncol=1)
    fig.suptitle("Few-shot effect, paired on identical items", fontsize=13,
                 weight="bold", y=1.03)
    save(fig, "fig5_fewshot")

# --- fig6: per-class heatmap for all-11 --------------------------------------
d = perclass[(perclass.task == "subtype_all")]
d = d[d.model.isin(REP[REP.task == "subtype_all"].model)]
# ONE regime per row. perclass carries encoder rows for both in_domain and
# cross_project, and the max() in the pivot below took the better of the two
# PER CLASS - 3 of the 22 encoder cells came from the cross-project arm, one
# of them 0.14 above its in-domain value - producing a row that belongs to no
# single experiment under a heading that names none. Encoders enter in-domain
# (the arm fig4 and tab1 show); prompted models have one regime anyway.
d = d[d.eval_regime.isin(IN_DOMAIN)]
if len(d):
    piv = d.pivot_table(index="model", columns="category", values="f1", aggfunc="max")
    # Support differs by row: the encoders score subtype_all on 524 items and
    # the LLMs on 491, so a single max() printed the ENCODER's n above every
    # column, including the six LLM rows underneath. Show the range when the
    # rows disagree rather than one row's number for all of them.
    _sup_lo = d.groupby("category").support.min()
    _sup_hi = d.groupby("category").support.max()
    piv = piv[[c for c in CATEGORIES_ALL if c in piv.columns]]
    fig, ax = plt.subplots(figsize=(1.05 * piv.shape[1] + 3, 0.55 * len(piv) + 2.4))
    im = ax.imshow(piv.values, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(piv.shape[1]))
    ax.set_xticklabels(
        [f"{c.replace('_',' ')}\n(n={int(_sup_lo.get(c, 0))})"
         if int(_sup_lo.get(c, 0)) == int(_sup_hi.get(c, 0))
         else f"{c.replace('_',' ')}\n(n={int(_sup_lo.get(c, 0))}-{int(_sup_hi.get(c, 0))})"
         for c in piv.columns], rotation=45, ha="right", fontsize=8.5)
    ax.set_yticks(range(len(piv)))
    ax.set_yticklabels(piv.index, fontsize=9)
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5,
                        color="white" if v > 0.55 else "#111827")
    ax.set_title("Per-class F1, NFR sub-types (all-11)", fontsize=12, weight="bold")
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.8, label="F1")
    save(fig, "fig6_perclass_heatmap")


# ============================================================================
# provenance
# ============================================================================
prov = {
    "sources": provenance,
    "n_rows": int(len(store)),
    "gates": {"min_class_support": MIN_CLASS_SUPPORT, "min_n": MIN_N,
              "min_parse_rate": MIN_PARSE_RATE, "alpha": ALPHA,
              "bootstrap": f"stratified, {N_BOOT} replicates",
              "scoring": "strict (unparseable answers counted as incorrect)"},
    "head_rows": int(len(head)),
    "head_encoder_rows": _n_enc,
    "head_prompted_rows": _n_llm,
    "cells_total": int(len(metrics)),
    "cells_reportable": int(metrics.reportable.sum()),
    "cells_reportable_encoder": int((REP.tier == "finetuned").sum()),
    "cells_excluded": metrics[~metrics.reportable][
        ["model", "task", "dataset", "eval_regime", "n", "exclusion_reason"]
    ].to_dict("records"),
    "mcnemar_tests": int(len(mcn)),
    "like_for_like_rows": int(len(likeforlike)),
    "per_project_cells": int(len(perproject)),
    "per_project_spread_rows": int(len(spread)),
    "y_pred_strict_adopted": bool("y_pred_strict" in store.columns),
    "core_restricted_comparisons": bool(store.in_core.any()
                                        and not store.in_core.all()),
    "models": sorted(store.model.unique()),
    "tasks": sorted(store.task.unique()),
    "regimes": sorted(store.eval_regime.unique()),
}
json.dump(prov, open(f"{OUT}/stage4_provenance.json", "w"), indent=2)

print("\n[done] outputs:")
for f in sorted(glob.glob(f"{OUT}/tab*") + glob.glob(f"{OUT}/fig*") +
                glob.glob(f"{OUT}/stage4_provenance.json")):
    print(f"   {os.path.basename(f):38s} {os.path.getsize(f)/1024:8.1f} KB")


ATTACHED INPUTS

unified.parquet   <- Stage 1 corpus        (Stages 2, 4)
   /kaggle/input/datasets/zahrasamir/stage1-outputs/unified.parquet
            95,290 bytes   2026-08-27 08:52  >> THIS ONE WILL BE USED

splits.json   <- Stage 1 splits        (Stage 2)
   /kaggle/input/datasets/zahrasamir/stage1-outputs/splits.json
         1,454,089 bytes   2026-08-27 08:52  >> THIS ONE WILL BE USED

predictions_finetuned.parquet   <- Stage 2 encoder store (Stage 2 resume, 4, 5)
   /kaggle/input/datasets/zahrasamir/stage2-outputs/predictions_finetuned.parquet
           210,255 bytes   2026-08-27 08:52  >> THIS ONE WILL BE USED

predictions_llm.parquet   <- Stage 3 LLM binary    (Stages 4, 5)
   (not attached)

predictions_subtype.parquet   <- Stage 3b raw subtype  (Stage 3b-repair)
   (not attached)

predictions_subtype_repaired.parquet   <- Stage 3b repaired    (Stages 4, 5)
   /kaggle/input/notebooks/zahrasamir/stage3b-repair/predictions_subtype_repaired.parquet
           428,497 bytes   